# CS224N L01 — 代码胶囊：共现矩阵 + SVD 降维

**Waypoint WP02**：分布语义（Distributional Semantics）  
**核心概念**：从语料构建共现矩阵（co-occurrence matrix），用 SVD 降维得到词向量  
**官方锚点**：Notes §3.1 (co-occurrence matrices); A1 Part 1 Q1.1-1.3

---

## 这段代码在看什么

这个 notebook 实现了 CS224N Notes §3.1 描述的核心流程：

1. **构建共现矩阵**：统计每个词在滑动窗口内和哪些词一起出现过
2. **Log 归一化**：用 log(1+x) 压缩高频词的影响
3. **SVD 降维**：把高维稀疏矩阵压缩到低维密集向量空间
4. **余弦相似度**：衡量降维后词向量的方向相似度
5. **窗口对比**：比较不同窗口大小对语义捕捉的影响

> [!info] 为什么不能只读文字？
> Notes §3.1 描述了共现矩阵的构建过程，但只有实际构建一个小矩阵并做 SVD，
> 才能理解窗口大小和降维维度的效果——为什么同主题的 banking 和 money 向量接近，
> 而 2D 降维为什么不足以完全分开两个主题。

## 1. 导入库

只需要 numpy（矩阵运算）和 matplotlib（可视化）。Colab 自带，无需额外安装。

In [ ]:
# numpy 用于矩阵构建和 SVD 计算
import numpy as np
# matplotlib 用于可视化共现矩阵热力图和 2D 散点图
import matplotlib
matplotlib.use('Agg')  # 无头模式，Colab 和本地都能用
import matplotlib.pyplot as plt

## 2. 定义 Toy Corpus

我们设计了一个小语料，包含两个清晰的语义簇：

- **Finance（金融）**：banking, money, finance, economy, market, invest
- **Nature（自然）**：river, lake, forest, mountain, valley, ocean

另外还有 2 句"桥接句"，让两个簇之间有少量联系。

> [!tip] 中文扶手
> 分布语义学的核心思想（Firth 名言）："You shall know a word by the company it keeps"
> ——一个词的含义由它的上下文决定。
> banking 总出现在 money/finance 附近，river 总出现在 lake/forest 附近，
> 所以共现矩阵能捕捉这种"语义邻居"关系。

In [ ]:
# 定义 14 个小句子作为语料
# Finance 簇（6 句）：banking/money/finance/economy/market/invest 经常共现
# Nature 簇（6 句）：river/lake/forest/mountain/valley/ocean 经常共现
# 桥接句（2 句）：跨簇共享个别词（如 flows, invest）
CORPUS = [
    # Finance sentences（金融主题）
    "the banking system manages money and finance",
    "the economy depends on banking and finance",
    "money flows through the banking system",
    "the economy and finance are connected to the market",
    "invest in the banking market and finance",
    "the market economy depends on money and invest",
    # Nature sentences（自然主题）
    "the river flows through the forest and valley",
    "the lake is near the mountain and forest",
    "a mountain rises above the lake and valley",
    "the forest surrounds the lake and river",
    "the river and lake are in the valley",
    "a mountain overlooks the river and ocean",
    # Bridge sentences（桥接句，跨主题）
    "the river flows like money through the economy",
    "invest in the ocean and the market",
]

print(f"语料共 {len(CORPUS)} 个句子")

## 3. 构建词汇表

把语料中所有不重复的词排序，建立 词→索引 的映射。
这个索引决定了共现矩阵的行和列分别对应哪个词。

In [ ]:
# 分词：简单按空格切分 + 小写化
def tokenize(text):
    return text.lower().split()

# 收集所有 token，构建排序后的词汇表
all_tokens = []
for sent in CORPUS:
    all_tokens.extend(tokenize(sent))

vocab_set = sorted(set(all_tokens))        # 排序后的不重复词列表
word2idx = {w: i for i, w in enumerate(vocab_set)}  # 词 → 矩阵行/列索引
idx2word = {i: w for w, i in word2idx.items()}       # 索引 → 词（反查用）
V = len(vocab_set)

print(f"总 token 数: {len(all_tokens)}")
print(f"词汇表大小 |V| = {V}")
print(f"词汇表: {vocab_set}")

## 4. 构建共现矩阵（对应 Notes §3.1 核心步骤）

> [!info] Notes §3.1 的构建步骤
> 1. 确定词汇表 V
> 2. 建 |V|×|V| 的零矩阵 M
> 3. 遍历语料，对每个中心词，统计窗口内每个上下文词的出现次数
> 4. M[中心词, 上下文词] += 1

我们用 `window_size=2`，即中心词左右各看 2 个位置。
例如 "the banking system manages money"，以 banking 为中心，窗口内是 the, system, manages, money。

In [ ]:
# 构建对称共现矩阵
# 对每个中心词 center，在 [i-window, i+window] 范围内统计上下文词 ctx 的出现次数
def build_cooccurrence(corpus, word2idx, window_size):
    V = len(word2idx)
    cooc = np.zeros((V, V), dtype=int)  # |V|×|V| 零矩阵
    
    for sent in corpus:
        tokens = tokenize(sent)
        for i, center in enumerate(tokens):
            c_idx = word2idx[center]           # 中心词的索引
            start = max(0, i - window_size)     # 窗口左边界
            end = min(len(tokens), i + window_size + 1)  # 窗口右边界
            for j in range(start, end):
                if i == j:
                    continue  # 跳过中心词自己
                ctx = tokens[j]
                ctx_idx = word2idx[ctx]
                cooc[c_idx, ctx_idx] += 1     # 共现计数 +1
    
    return cooc

# 用 window=2 构建共现矩阵
WINDOW = 2
cooc_raw = build_cooccurrence(CORPUS, word2idx, window_size=WINDOW)

print(f"--- 共现矩阵 (window={WINDOW}) ---")
print(f"矩阵形状: {cooc_raw.shape}")
print(f"共现总计数: {cooc_raw.sum()}")
print(f"非零元素数: {np.count_nonzero(cooc_raw)}")
print(f"稀疏度: {1 - np.count_nonzero(cooc_raw) / cooc_raw.size:.1%}")

## 运行后先看哪里

> [!tip] 先看共现矩阵的子矩阵
> 下面只显示内容词（去掉 the/a/is 等功能词）的子矩阵。
> 
> **关键观察**：
> - banking 行中，system(2)、finance(1)、money(0) —— banking 和 system 共现最多
> - river 行中，flows(2)、lake(2) —— river 和 flows/lake 共现最多
> - 跨簇的格子（如 banking-river）大多是 0 —— 两个主题的词很少在同一个窗口出现
> 
> 这就是"分布语义"的直觉：**语义相似的词有相似的共现模式**。

In [ ]:
# 只显示内容词（去掉功能词）的子矩阵，便于观察
content_words = [w for w in vocab_set if w not in {'the', 'a', 'is', 'in', 'of', 'and', 'to', 'with'}]
cw_indices = [word2idx[w] for w in content_words]

print("共现子矩阵（仅内容词）：")
header = "          " + "".join(f"{w:>10s}" for w in content_words)
print(header)
for i, w in enumerate(content_words):
    row_vals = [cooc_raw[cw_indices[i], cw_indices[j]] for j in range(len(content_words))]
    row = f"{w:>10s}" + "".join(f"{v:10d}" for v in row_vals)
    print(row)

print()
print("关键词的非零共现（window=2）：")
key_words = ['banking', 'money', 'finance', 'river', 'lake', 'forest']
for w in key_words:
    idx = word2idx[w]
    neighbors = [(idx2word[j], int(cooc_raw[idx, j])) for j in range(V) if cooc_raw[idx, j] > 0]
    neighbors.sort(key=lambda x: -x[1])
    print(f"  {w:>10s}: {neighbors[:6]}")  # 只显示 top 6

## 5. Log 归一化（Notes §3.1 推荐）

> [!tip] Notes §3.1 原文
> "Taking the log token frequency ends up being much more useful"

**为什么需要 log 归一化？**
- 原始共现次数中，功能词（the/and）的计数非常高（the 和 banking 共现 3 次）
- 这些高频词会主导 SVD 的方向，让降维结果偏向语法而非语义
- `log(1+x)` 压缩极端值：max 从 9 压到 2.30，抑制了高频词的主导

> [!warning] 容易误解的地方
> log 归一化不是万能的。它只是第一步，实际中还会用 PPMI（Positive PMI）
> 等更高级的方法来进一步去除偏差。

In [ ]:
# log(1+x) 归一化：压缩极端值，让 SVD 更关注语义而非频率
cooc_log = np.log1p(cooc_raw.astype(float))

print("--- Log 归一化效果 ---")
print(f"归一化前: max = {cooc_raw.max()}, 非零均值 = {cooc_raw[cooc_raw > 0].mean():.2f}")
print(f"归一化后: max = {cooc_log.max():.4f}, 非零均值 = {cooc_log[cooc_log > 0].mean():.4f}")
print()
print("→ 最大值从 9 压缩到 2.30，高频词的主导被抑制")

## 6. SVD 降维（核心步骤）

用 [[Lectures/L01-word-vectors/00-concept-glossary#svd|SVD]] 把 |V|×|V| 的共现矩阵压缩到 |V|×2。

> [!info] SVD 的直觉
> SVD 找到共现矩阵中"最重要的两个方向"——共现模式中方差最大的两个轴。
> 语义相似的词在这些轴上的投影会接近。
> 
> 奇异值的大小代表每个方向捕获了多少信息。
> 前 2 个奇异值解释的方差比例告诉我们 2D 保留了多少原始信息。

In [ ]:
# SVD 降维：保留前 k 个奇异值/向量
# U[:, :k] * S[:k] 得到每个词的 k 维向量表示
def svd_reduce(matrix, k=2):
    U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
    reduced = U[:, :k] * S[:k]  # 形状 (V, k)
    return reduced, S

k = 2
embeddings, singular_values = svd_reduce(cooc_log, k=k)

# 查看奇异值和方差解释率
explained_var = (singular_values[:k]**2 / (singular_values**2).sum() * 100)
print(f"--- SVD 降维到 {k}D ---")
print(f"Log 归一化后的奇异值 (top 6): {singular_values[:6].round(4)}")
print(f"前 {k} 个奇异值解释的方差: {explained_var.round(1)}%")
print()
print("→ 2D 只保留了约 58% 的信息，42% 被丢弃——这就是为什么后面会看到簇分离不完美")
print()

# 显示内容词的 2D 坐标
print("内容词的 2D 坐标（log 归一化 SVD）：")
for w in content_words:
    i = word2idx[w]
    print(f"  {w:>10s}: [{embeddings[i, 0]:8.4f}, {embeddings[i, 1]:8.4f}]")

## 7. 余弦相似度

用 [[Lectures/L01-word-vectors/00-concept-glossary#cosine-similarity|cosine similarity]] 衡量降维后词向量的方向相似度。

- 值接近 1：两个词方向几乎相同（语义相似）
- 值接近 0：两个词方向正交（语义无关）
- 值接近 -1：两个词方向相反

> [!tip] 输出怎么解释
> 看下面的结果，注意对比：
> - **同簇词对**（如 banking-money, river-lake）的 cosine 应该高
> - **跨簇词对**（如 banking-river）的 cosine 应该低
> 
> 但实际结果可能出乎意料——2D 降维太激进时，跨簇词也可能很接近！

In [ ]:
# 余弦相似度：衡量两个向量方向的相似程度
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 对比同簇 vs 跨簇的词对
pairs = [
    ("banking", "finance",   "同簇 (finance)"),
    ("banking", "money",     "同簇 (finance)"),
    ("banking", "economy",   "同簇 (finance)"),
    ("river",   "lake",      "同簇 (nature)"),
    ("river",   "forest",    "同簇 (nature)"),
    ("lake",    "valley",    "同簇 (nature)"),
    ("banking", "river",     "跨簇"),
    ("finance", "mountain",  "跨簇"),
    ("money",   "ocean",     "跨簇"),
]

print("--- 余弦相似度 (log 归一化 SVD, 2D) ---")
for w1, w2, label in pairs:
    sim = cosine_sim(embeddings[word2idx[w1]], embeddings[word2idx[w2]])
    print(f"  cos({w1:>10s}, {w2:>10s}) = {sim:.4f}  [{label}]")

## 8. 窗口大小的影响

> [!tip] Notes §3.1 关键观察
> - **短窗口**（1 词）→ 捕捉**语法**属性（名词旁常出现 the/is/and）
> - **中窗口**（2-3 词）→ 平衡语法和语义
> - **大窗口**（5+ 词或文档级）→ 捕捉**语义/主题**属性

试试不同窗口大小对相似度的影响：

In [ ]:
# 对比 window=1/2/3 对关键词对 cosine 的影响
print(f"{'Window':>6s}  {'banking-money':>14s}  {'river-lake':>14s}  {'banking-forest':>15s}")
print("-" * 55)
for ws in [1, 2, 3]:
    cooc_ws = np.log1p(build_cooccurrence(CORPUS, word2idx, window_size=ws).astype(float))
    emb_ws, _ = svd_reduce(cooc_ws, k=k)
    
    sim_bm = cosine_sim(emb_ws[word2idx["banking"]], emb_ws[word2idx["money"]])
    sim_rl = cosine_sim(emb_ws[word2idx["river"]], emb_ws[word2idx["lake"]])
    sim_bf = cosine_sim(emb_ws[word2idx["banking"]], emb_ws[word2idx["forest"]])
    
    print(f"  w={ws}  {sim_bm:>14.4f}  {sim_rl:>14.4f}  {sim_bf:>15.4f}")

print()
print("→ 输出怎么解释：")
print("  w=1: 所有 cosine 接近 1.0 —— 语法角色主导（名词旁都是 the/is/and）")
print("  w=2: 同簇略高于跨簇 —— 开始捕捉语义，但 2D 仍然太压缩")
print("  w=3: banking-forest 降到 0.94 —— 更大窗口让主题差异更明显")

## 9. 可视化：总览

三张图并排：
- **左**：共现矩阵热力图（内容词子矩阵）
- **中**：SVD 2D 词向量散点图
- **右**：窗口大小对 cosine 的影响

In [ ]:
# 三合一可视化
finance_words = ['banking', 'money', 'finance', 'economy', 'market', 'invest']
nature_words = ['river', 'lake', 'forest', 'mountain', 'valley', 'ocean']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 左：共现矩阵热力图（内容词）
sub_matrix = cooc_raw[np.ix_(cw_indices, cw_indices)]
im = axes[0].imshow(sub_matrix, cmap='YlOrRd', aspect='auto')
axes[0].set_title(f'共现矩阵 (window={WINDOW}, 内容词)', fontsize=11)
axes[0].set_xticks(range(len(content_words)))
axes[0].set_yticks(range(len(content_words)))
axes[0].set_xticklabels(content_words, rotation=90, fontsize=8)
axes[0].set_yticklabels(content_words, fontsize=8)
plt.colorbar(im, ax=axes[0], fraction=0.046)

# 中：2D 散点图
for w in content_words:
    i = word2idx[w]
    x, y = embeddings[i, 0], embeddings[i, 1]
    if w in finance_words:
        color, marker = '#e74c3c', 'o'    # 红圆 = Finance
    elif w in nature_words:
        color, marker = '#2ecc71', 's'    # 绿方 = Nature
    else:
        color, marker = '#f39c12', 'D'    # 橙菱 = 其他
    axes[1].scatter(x, y, c=color, marker=marker, s=100, zorder=5, 
                    edgecolors='black', linewidth=0.5)
    axes[1].annotate(w, (x, y), fontsize=9, fontweight='bold', 
                     ha='center', va='bottom', xytext=(0, 6), textcoords='offset points')

axes[1].set_title(f'SVD 2D 词向量 (log 归一化, window={WINDOW})', fontsize=11)
axes[1].set_xlabel('SVD 分量 1')
axes[1].set_ylabel('SVD 分量 2')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5, linestyle='--')
axes[1].axvline(x=0, color='k', linewidth=0.5, linestyle='--')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Finance'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#2ecc71', markersize=10, label='Nature'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#f39c12', markersize=10, label='其他'),
]
axes[1].legend(handles=legend_elements, fontsize=9, loc='upper right')

# 右：窗口对比柱状图
window_sizes = [1, 2, 3]
all_sims = {}
for ws in window_sizes:
    cooc_ws = np.log1p(build_cooccurrence(CORPUS, word2idx, window_size=ws).astype(float))
    emb_ws, _ = svd_reduce(cooc_ws, k=k)
    all_sims[ws] = (
        cosine_sim(emb_ws[word2idx["banking"]], emb_ws[word2idx["money"]]),
        cosine_sim(emb_ws[word2idx["river"]], emb_ws[word2idx["lake"]]),
        cosine_sim(emb_ws[word2idx["banking"]], emb_ws[word2idx["forest"]]),
    )

x_pos = np.arange(len(window_sizes))
width = 0.25
axes[2].bar(x_pos - width, [all_sims[ws][0] for ws in window_sizes], width, 
            label='banking-money\n(同簇)', color='#e74c3c', alpha=0.8)
axes[2].bar(x_pos, [all_sims[ws][1] for ws in window_sizes], width, 
            label='river-lake\n(同簇)', color='#2ecc71', alpha=0.8)
axes[2].bar(x_pos + width, [all_sims[ws][2] for ws in window_sizes], width, 
            label='banking-forest\n(跨簇)', color='#3498db', alpha=0.8)
axes[2].set_title('窗口大小对余弦相似度的影响', fontsize=11)
axes[2].set_xlabel('窗口大小')
axes[2].set_ylabel('余弦相似度')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f'w={ws}' for ws in window_sizes])
axes[2].legend(fontsize=8)
axes[2].set_ylim(-0.5, 1.1)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('Labs/L01-word-vectors/outputs/co-occurrence-matrix-overview.png', dpi=150, bbox_inches='tight')
plt.close()
print("已保存: Labs/L01-word-vectors/outputs/co-occurrence-matrix-overview.png")

## 10. 详细 2D 散点图（带聚类区域）

> [!tip] 读图指南
> - **红圆（Finance）**：banking, money, finance, economy, market, invest
> - **绿方（Nature）**：river, lake, forest, mountain, valley, ocean
> - 椭圆标注了两个簇的大致范围
> 
> **关键观察**：两组词在 2D 空间中**并未完全分开**——大部分词都聚集在左下方。
> 这正是 2D 降维信息损失的表现：前 2 个奇异值只解释了 58% 的方差。

In [ ]:
# 详细 2D 散点图，带聚类区域椭圆
from matplotlib.patches import Ellipse

fig2, ax2 = plt.subplots(figsize=(10, 8))

# Finance 簇的椭圆标注
fin_pts = np.array([embeddings[word2idx[w]] for w in finance_words])
fin_center = fin_pts.mean(axis=0)
fin_std = fin_pts.std(axis=0) + 0.3
ellipse_fin = Ellipse(fin_center, fin_std[0]*4, fin_std[1]*4, 
                       alpha=0.12, color='#e74c3c', label='Finance 簇')
ax2.add_patch(ellipse_fin)

# Nature 簇的椭圆标注
nat_pts = np.array([embeddings[word2idx[w]] for w in nature_words])
nat_center = nat_pts.mean(axis=0)
nat_std = nat_pts.std(axis=0) + 0.3
ellipse_nat = Ellipse(nat_center, nat_std[0]*4, nat_std[1]*4, 
                       alpha=0.12, color='#2ecc71', label='Nature 簇')
ax2.add_patch(ellipse_nat)

# 绘制所有内容词
for w in content_words:
    i = word2idx[w]
    x, y = embeddings[i, 0], embeddings[i, 1]
    if w in finance_words:
        color, marker, sz = '#e74c3c', 'o', 120
    elif w in nature_words:
        color, marker, sz = '#2ecc71', 's', 120
    else:
        color, marker, sz = '#f39c12', 'D', 80
    ax2.scatter(x, y, c=color, marker=marker, s=sz, zorder=5, 
                edgecolors='black', linewidth=0.5)
    ax2.annotate(w, (x, y), fontsize=10, fontweight='bold', 
                 ha='center', va='bottom', xytext=(0, 8), textcoords='offset points')

ax2.set_title(f'共现矩阵 + SVD: 2D 词向量\n(log 归一化, window={WINDOW}, k={k})\nFinance（红）vs Nature（绿）', fontsize=13)
ax2.set_xlabel('SVD 分量 1', fontsize=11)
ax2.set_ylabel('SVD 分量 2', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='k', linewidth=0.5, linestyle='--')
ax2.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
ax2.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('Labs/L01-word-vectors/outputs/co-occurrence-matrix-2d-embeddings.png', dpi=150, bbox_inches='tight')
plt.close()
print("已保存: Labs/L01-word-vectors/outputs/co-occurrence-matrix-2d-embeddings.png")

## 和本讲哪个 waypoint 对应

| 课堂材料 | 对应内容 |
|----------|----------|
| Slides p21 | Firth 名言 "You shall know a word by the company it keeps" |
| Slides p22-23 | Context window 和 dense vectors 的概念 |
| Notes §3.1 | Co-occurrence matrix 构建、log frequency weighting、SVD 降维 |
| A1 Part 1 Q1.1-1.3 | 实现 co-occurrence matrix + SVD（本 capsule 的简化版） |

本 capsule 对应 **WP02（分布语义）**，是理解 Word2Vec 之前的重要基础。

## 容易误解的地方

> [!warning] 注意这些常见误区
> 
> 1. **2D cosine 高不等于语义相似**：banking-river=0.998 是因为 2D 太压缩，所有向量方向趋同。高维空间中它们会分开。
> 
> 2. **Log 归一化不是万能的**：它抑制了高频词但仍然不够——实际中还会用 PPMI（Positive PMI）等更高级的方法。
> 
> 3. **功能词主导共现**：the/and 的共现次数最高（the 和 banking 共现 3 次）。这就是为什么需要 log 归一化或过滤功能词。
> 
> 4. **SVD vs Word2Vec**：SVD 是计数方法（先统计共现再降维）；Word2Vec 是预测方法（直接训练神经网络）。两者思路不同但都能得到有意义的词向量。
> 
> 5. **方差解释率**：前 2 奇异值只解释 58.1%。如果要保留 90% 方差，需要约 10 个维度（本语料）。真实语料需要更多。

## 小结

> [!info] 学到了什么
> 1. **共现矩阵**：统计词在窗口内共同出现的次数，捕捉分布语义
> 2. **Log 归一化**：压缩高频词的影响，让 SVD 更有效
> 3. **SVD 降维**：把高维稀疏矩阵压缩到低维密集向量
> 4. **窗口大小**：短窗口捕捉语法，长窗口捕捉语义/主题
> 5. **Cosine similarity**：衡量词向量方向相似度
> 6. **2D 的局限**：2D 降维太激进，跨簇词也会被压到一起——实际中用 50-300 维